# This noteboook aims to do an initial analysis of ERA5 and ERA5land Tmax data for identifying heatwave events over europe

This notebook prepares all **inputs** and **labels** required for training the deep learning (DL) model.
It processes two complementary types of climate data:

---

## Local-Scale (Site-Based) Data Pipeline
Local variables extracted at individual study sites, such as:

- `swvl1`, `swvl2`, `swvl3` (soil water)
- drought indicators (`SPEI`, `SPI`)
- local temperature (`tasmax`, `tasmin`)

This pipeline produces:

- **standardized anomalies**
- **daily extreme-event labels (0/1)**
- **lagged versions** of all variables (final DL inputs)

## Large-Scale (Gridded) Data Pipeline
High-dimensional atmospheric fields:

- `g500`, `g200` (geopotential height)
- `psl` (mean sea level pressure)
- any additional ERA5 fields

This pipeline mirrors the local-scale processing but applied to spatial grids:
- detrending
- anomaly computation
- lagged feature creation
- **regridding** (0.25° → 1°) to reduce model complexity

## Deep Learning Labels: Extreme Temperature Classification

Labels consist of a **binary daily indicator** for temperature extremes.
A day is labeled *1* if `tasmax` exceeds a **smoothed 90th percentile** computed over a reference climatology period, following:

**Perkins & Alexander (2013)**
*On the Measurement of Heat Waves*
https://journals.ametsoc.org/view/journals/clim/26/13/jcli-d-12-00383.1.xml

Otherwise, the day is labeled *0*.

---

# Structure of the Notebook

The notebook is organized into blocks that process **local-scale** and **large-scale** data.
Local-scale blocks come first; large-scale blocks follow.

---

### 1. Compute Standardized Anomalies (Local-Scale)
Computes standardized anomalies relative to the **1950–2000** reference climatology using LOESS-smoothed seasonal cycles.
Outputs: NetCDF files containing the anomalies for each study site.

### 2. Detrending (Local-Scale)
Optional block for removing linear trends from site-based variables.
*(Not used in the paper but included for reproducibility.)*

### 3. Extreme-Event Detection (Local-Scale)
Core block for creating the DL labels.

Includes:
- computing climatology and percentiles
- LOESS smoothing
- heatwave detection using a moving window
- saving binary extreme-event masks
- generating summary metrics (duration, frequency, intensity)

Outputs per site:
- NetCDF file with anomalies + extreme labels
- text file with heatwave statistics


### 4. Creation of Lagged Features (Local-Scale)
Generates lagged versions (e.g., 1–7 days) of anomalies and extreme classifications.
These lagged predictors form the **final input features** for the neural network.

Outputs:
- NetCDF file with anomalies + labels + lagged variables

### 5. Detrending (Large-Scale)
Optional linear detrending of full ERA5 fields.

### 6. Anomaly Computation (Large-Scale)
Computes standardized anomalies for atmospheric fields on the original 0.25° grid.

### 7. Regridding
Regrids all large-scale fields from **0.25° × 0.25°** to **1° × 1°**
to reduce storage size and the number of parameters required by the DL model.

### 8. Lagged Feature Generation (Regridded Large-Scale)
Creates multi-day lagged versions of the regridded anomalies.
Outputs are the final large-scale predictors used for DL experiments.

---

# Summary

This notebook generates the complete dataset for heatwave prediction using deep learning:

### **Inputs**
- local-scale standardized anomalies
- large-scale standardized anomalies
- lagged versions of all variables
- (optional) detrended versions
- regridded large-scale fields

### **Labels**
- binary temperature extreme classification per day
- metrics describing heatwave behaviour

All results are saved as **NetCDF files** for easy model ingestion.


In [3]:
# Import of used libraries ------------------------------------------------------------------------------------------------------------

import numpy as np
from matplotlib import pyplot as plt
from netCDF4 import Dataset as ncread
import xarray as xr
from scipy.stats import linregress
from datetime import datetime, timedelta
import pandas as pd
import cartopy
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import cartopy.mpl.ticker as cticker
import os
import re
import ast
import xesmf as xe
from matplotlib.backends.backend_pdf import PdfPages
import sys
from tools import lagged_data, regrid
from tools.clim import Compute_anomalies, Compute_window_percentile_reference_period, detect_HW
from tools.heatwaves import Compute_metrics
from tools.loess import loess_ts

ModuleNotFoundError: No module named 'numpy'

LOESS fit is passed to the percentile values based on Mahlstein et al. 2015

---
# Local Scale Data
---

#### Global variables definition:
Use the following variables to define the paths in which you want to save the files generated in each step, and also the variables for which you would want to compute each step.


In [ ]:
sites = ['cordoba','hannover','stockholm','lyon','belgrado','marrakech']

# Local-scale variables
local_vars = ['tasmax','tasmin','swvl1','swvl2','swvl3']
variable_extreme = 'tasmax'

# Reference period for climatology
REF_START = '1950'
REF_END = '2000'
FULL_START = '1950'
FULL_END = '2023'

# Percentile threshold
PERCENTILE = 0.9
percentile_str = "90p"

# Paths
input_path_anomalies = '/gpfs/scratch/bsc32/bsc167965/data/era5_land/'
output_path_anomalies  = "/gpfs/scratch/bsc32/bsc167965/data/era5_land/anomalies/"

input_path_heatwave = '/gpfs/projects/bsc32/bsc167965/observational_TX/spei_tasmax_{site}_hg_3.nc'
output_path_heatwave  = '/gpfs/projects/bsc32/bsc214253/labels'
output_path_metrics = 'HW_dates_detected_and_metrics/'

input_path_lagged = output_path_heatwave
output_path_lagged = '/gpfs/scratch/bsc32/214253/data_test/localScale_data_test/'

# Variables definition and their lags
variables_lagged = [
 'tasmax_anomalies',
 'tasmin_anomalies',
 'swvl1_anomalies',
 'swvl2_anomalies',
 'swvl3_anomalies',
 'tasmax_extreme_classification']

# Dictionary indicating number of days of lagged-data for each of the variables
lag_dict = {
 'tasmax_anomalies': [1, 2, 3],
 'tasmin_anomalies': [1, 2, 3],
 'swvl1_anomalies': [1, 2, 3, 4, 5, 6, 7],
 'swvl2_anomalies': [1, 2, 3, 4, 5, 6, 7],
 'swvl3_anomalies': [1, 2, 3, 4, 5, 6, 7],
 'tasmax_extreme_classification': [1,2,3] }

## 1. Compute and Save Standardized Anomalies for Local-Scale Variables

This block processes the ERA5-Land variables for each study site and computes **standardized anomalies** for the model input. The anomalies are calculated relative to a **reference climatology period** (1950–2000) and smoothed using LOESS. The resulting anomalies are saved in new NetCDF files for downstream use in the deep learning model.

### Steps:

1. **Define output directory**
   - All processed files with anomalies will be saved in `outputpath`.


2. **Loop over study sites**
   - For each site in `SITES`, open the corresponding NetCDF dataset containing ERA5-Land variables (1950–2024).


3. **Loop over local-scale variables**
   - Variables like `swvl1`, `swvl2`, `swvl3` are processed.
   - Compute standardized anomalies using the `Compute_anomalies` function:
     - **Reference period**: 1950–2000.
     - **LOESS smoothing**: applied to the climatology to remove seasonal noise.
     - **Standardization**: each anomaly is scaled as  $$standarized_{anomaly} = \frac{anomaly-\mu}{\sigma}$$
       where μ and σ are the mean and standard deviation of the climatology.


4. **Store anomalies in the dataset**
   - Each anomaly variable is saved with descriptive metadata:
     - `long_name`: `"Anomalies with LOESS climatology"`
     - `description`: Explanation of the computation method and reference period.


5. **Save dataset to NetCDF**
   - The processed dataset (original variables + anomalies) is written to a new NetCDF file.
   - Filenames clearly indicate the site and time period, e.g.,
     `"_variables_era5land_data_with_anomalies_cordoba_1950_2024.nc"`.


6. **Memory management**
   - Optionally close the dataset after saving to free memory.



In [5]:
# Loop over all the sites for which you want to process the data
for site in sites:
    # Construct the path to the input NetCDF file for the current site
    # This file should contain the original ERA5-Land variables for the period 1950–2024
    filename = f'variables_era5land_data_{site}_1950_2024.nc'

    # Open the dataset using xarray
    ds = xr.open_dataset(input_path_anomalies + filename)

    # Loop over all local-scale variables to compute anomalies
    for variable in local_vars:
        # Compute standardized anomalies for the given variable
        # Using '1950'-'2000' as the reference period for climatology
        # Compute_anomalies applies LOESS smoothing to the climatology and standardizes
        standarized_anomalies = Compute_anomalies(ds, variable, '1950', '2000')

        # Save the standardized anomalies back into the dataset with a descriptive name
        ds[f'{variable}_anomalies'] = standarized_anomalies
        ds[f'{variable}_anomalies'].attrs = {
            'long_name': 'Anomalies with LOESS climatology',
            'description': (
                'Standardized anomalies computed using LOESS-smoothed climatology '
                'over the reference period 1950–2000.'
            )
        }

    # Save the dataset with anomalies to a new NetCDF file
    # The filename indicates the site and period for clarity
    ds.to_netcdf(output_path_anomalies + f"_variables_era5land_data_with_anomalies_{site}_1950_2024.nc")

    # Optional: close the dataset to free memory
    ds.close()


## 2. Heatwave Detection and Anomaly Computation for ERA5-land data

This block performs the detection of heatwave events and the computation of soil water anomalies for multiple locations.  
The workflow includes the following steps:
1. Load ERA5-Land dataset.
2. Compute climatology, percentile thresholds, and climatological window.
3. Detect heatwave events using a percentile-based definition.
4. Compute anomalies for selected soil moisture variables.
5. Save anomalies + extreme-event flags to a NetCDF file.
6. Compute metrics of detected events and save them to a .txt report.

**Outputs generated per site:**  
- NetCDF file with anomalies and heatwave classification.  
- Text file with statistics of detected heatwave events.  


In [5]:
# ------------------------------------------------------------------------------
# USER SETTINGS
# ------------------------------------------------------------------------------
variables = ['swvl1', 'swvl2', 'swvl3']     # variables to compute anomalies for

# Ensure output directories exist
os.makedirs(output_path_anomalies, exist_ok=True)
os.makedirs(output_path_metrics, exist_ok=True)

for site in sites:

    print(f"\n--- Processing site: {site} ---")

    # -----------------------------
    # 1. Load dataset
    # -----------------------------
    ds_path = input_path_heatwave.format(site=site)
    ds = xr.open_dataset(ds_path).sel(time=slice('1950', '2023'))

    # -----------------------------
    # 2. Compute climatology, percentiles & reference window
    # -----------------------------
    # Uses custom function: Compute_window_percentile_reference_period()
    climatology, percentile, clim_window, full_window, loess_clim = Compute_window_percentile_reference_period(ds, variable_extreme, 0.9, 5, '1950', '2000')

    # Remove Feb 29 if present (leap day not compatible with 365-day climatology)
    if 366 in percentile.dayofyear:
        ds = ds.sel(time=~((ds.time.dt.month == 2) & (ds.time.dt.day == 29)))

    # -----------------------------
    # 3. Smooth percentile curve with LOESS
    # -----------------------------
    loess_percentile = loess_ts(percentile, na_rm=True, window=30, degree=1)

    # Convert result back to xarray for later use
    loess_percentile_xarray = xr.DataArray(
        loess_percentile,
        dims=["dayofyear"],
        coords={"dayofyear": percentile.dayofyear},
        name="loess_percentile",
    )

    # -----------------------------
    # 4. Detect heatwave events
    # -----------------------------
    HW_mask, HW_intensity, DateTime_detect = detect_HW(
        ds,
        variable_extreme,
        loess_percentile_xarray,
        3,
        site)

    # Make a writable copy
    ds_save = ds.copy()

    # Convert boolean mask to 0/1 integers
    HW_mask_0_1 = HW_mask.astype(int)

    # -----------------------------
    # 5. Compute anomalies for soil moisture variables
    # -----------------------------
    for var in variables:

        anomalies = Compute_anomalies(ds, var, '1950', '2000')

        # Save anomalies inside ds_save
        ds_save[f'{var}_anomalies'] = xr.DataArray(
            anomalies.values,
            coords={'time': ds.time},
            dims=['time']
        )
        ds_save[f'{var}_anomalies'].attrs = {
            'long_name': f'{var} standardized anomalies with LOESS climatology',
            'description': (
                f'These standardized anomalies for ERA5-Land {var} are computed from '
                'a raw climatology followed by LOESS smoothing.'
            )
        }

    # -----------------------------
    # 6. Save extreme-event classification to NetCDF
    # -----------------------------
    ds_save[f'{variable_extreme}_extreme_classification'] = (('time'), HW_mask_0_1)
    ds_save[f'{variable_extreme}_extreme_classification'].attrs = {
        'extreme detection method': f'Percentile-based ({percentile_str}), detrended data',
        'description': (
            "1 = extreme event, 0 = non-extreme. Heatwaves detected using smoothed "
            "LOESS percentile and 5-day moving window climatology. "
            "Percentile/climatology computed for 1971-2000 reference period."
        )
    }

    output_nc = os.path.join(
        output_path_heatwave,
        f'spei3_{site}_standardized_anomalies_and_extreme_detection.nc'
    )
    ds_save.to_netcdf(output_nc)
    print(f"Saved anomalies + event classification → {output_nc}")

    # -----------------------------
    # 7. Compute heatwave metrics
    # -----------------------------
    duration, frequency, freq_periods, max_int, mean_int, cum_int, percentages, counts = Compute_metrics(HW_mask, HW_intensity, DateTime_detect)

    # -----------------------------
    # 8. Write metrics to text report
    # -----------------------------
    output_txt = os.path.join(output_path_metrics, f'spei3_heatwave_stats_{site}.txt')

    with open(output_txt, 'w') as f:
        f.write(f"Site: {site}\n")
        f.write("Time period: 1950–2024\n")
        f.write(f"Reference period for percentile ({percentile_str}): 1971–2000\n")
        f.write("Minimum heatwave duration: 3 days\n\n")

        f.write(f"Max duration: {duration.max()}\n")
        f.write(f"Min duration: {duration.min()}\n")
        f.write(f"Total frequency: {frequency}\n")
        f.write(f"Frequency by period: {freq_periods}\n\n")

        f.write(f"Normal/extreme day counts by period: {counts}\n")
        f.write(f"Normal/extreme day percentages by period: {percentages}\n\n")

        f.write(f"Maximum intensity: {max_int.max()}\n")
        f.write(f"Mean intensity: {mean_int.max()}\n")
        f.write(f"Max accumulated intensity: {cum_int.max()}\n")
        f.write(f"Min accumulated intensity: {cum_int.min()}\n")

    print(f"Saved HW metrics → {output_txt}")

Heatwave events saved to 'heatwave_events_1950_2024_cordoba.csv'
All days 1950-2000 = 18615
{'1950-2000': {'HW_days': np.int64(1897), 'All_days': np.int64(18615)}, '1971-2000': {'HW_days': np.int64(1293), 'All_days': np.int64(10950)}, '2001-2024': {'HW_days': np.int64(1589), 'All_days': np.int64(8395)}}
{'1950-2000': {'HW_days_percent': np.float64(10.190706419554123)}, '1971-2000': {'HW_days_percent': np.float64(11.808219178082192)}, '2001-2024': {'HW_days_percent': np.float64(18.92793329362716)}}
{'1950-2000': 732, '1971-2000': 486, '2001-2024': 534}
Heatwave events saved to 'heatwave_events_1950_2024_hannover.csv'
All days 1950-2000 = 18615
{'1950-2000': {'HW_days': np.int64(1899), 'All_days': np.int64(18615)}, '1971-2000': {'HW_days': np.int64(1218), 'All_days': np.int64(10950)}, '2001-2024': {'HW_days': np.int64(1449), 'All_days': np.int64(8395)}}
{'1950-2000': {'HW_days_percent': np.float64(10.201450443190975)}, '1971-2000': {'HW_days_percent': np.float64(11.123287671232877)}, '20

IndexError: index 22 is out of bounds for axis 0 with size 22

## 3. Preparation of time-lagged features for ERA5-land
This block generates lagged versions of climate anomalies and extreme event classifications, preparing the data for machine learning (e.g., artificial neural networks).  

**Workflow:**

- For each site:
 - Load the NetCDF file containing standardized anomalies and extreme classification.
 - Call the `create_lagged_features_multiple()` function, which:
   - Iterates through each variable.
   - Creates new variables shifted by the specified lag days.
   - Stores them in the dataset with descriptive metadata.
 - Save the enriched dataset (original variables + lagged features) as a new NetCDF file.

**Outputs per site:**  
- NetCDF file containing the original anomalies, extreme classifications, and newly generated lagged features.  


In [3]:
# Loop over the different locations
for site in sites:
    # Path where inputs netCDF file is stored. Contains ERA5 land anomalies and extreme event classification based on tasmax
    input_filename = f'{percentile_str}_{site}_standarized_anomalies_and_extreme_detection.nc'
    ds = xr.open_dataset(input_path_lagged + input_filename)
    # Path where new netCDF file is saved
    output_filename = f'{percentile_str}_{site}_lagged_standarized_anomalies_and_extreme_detection.nc'
    # Apply the function
    ds_lagged = lagged_data.create_lagged_features_multiple(ds,variables_lagged,lag_dict,'lagged_era5_land_')
    ds_lagged.to_netcdf(output_path_heatwave + output_filename)

# {percentile_to_load}_{site}_lagged_standarized_anomalies_and_extreme_detection.nc <--- where the code originally loaded from, so this is our local_file

---
# Large-Scale data
---

#### Global variables definition:

Use the following variables to define the paths in which you want to save the files generated in each step, and also the variables for which you would want to compute each step.


In [5]:
# Directories in which the data will be saved:
input_path_regrid = '/gpfs/scratch/bsc32/bsc167965/data/era5/'
output_path_regrid = '/gpfs/scratch/bsc32/bsc167965/data/era5/regridded/'

input_path_anomalies = output_path_regrid
output_path_anomalies  = "/gpfs/scratch/bsc32/bsc167965/data/era5/anomalies/"

input_path_lagged = output_path_anomalies
output_path_lagged = '/gpfs/scratch/bsc32/214253/data_test/localScale_data_test/'

# List of variables for which to regrid the data
variables_regrid = ['g500', 'g200', 'psl']
# List of variables for which to compute anomalies
variables_anomalies = ['g200', 'psl']
# List of variables for which to create lagged features
variables_lagged = ['g500','g200','psl']
# Dictionary specifying the lag days for each variable
lag_dict = {'g200': [1, 2, 3],'g500': [1, 2, 3],'psl' : [1, 2, 3]}

## 1. Regridding ERA5 data

ERA5 data is originally provided at 0.25º x 0.25º resolution, which can be uncomfortable
for processing large datasets. To simplify downstream analysis, this script regrids
the selected atmospheric variables ('g500', 'g200', 'psl') to a 1º x 1º grid.

Workflow:
1. Load the ERA5 NetCDF file for each variable.
2. Apply a regridding function (`regrid.regridded_dataset`) to convert to 1º x 1º resolution.
3. Save the regridded dataset to a new NetCDF file.

This reduces file size and speeds up subsequent analysis while retaining sufficient spatial detail.

In [2]:
for variable in variables_regrid:
    input_filename = f'{variable}_era5_data_full_domain_1950_2024.nc'
    output_filename = f'{variable}_regridded_1x1_era5_data_full_domain_1950_2024.nc'

    # Load the ERA5 dataset for the current variable
    ds = xr.open_dataset(input_path_regrid + input_filename)

    # Regrid the dataset to 1º x 1º resolution
    ds_regridded = regrid.regridded_dataset(ds, variable)

    # Save the regridded dataset to a new NetCDF file
    ds_regridded.to_netcdf(output_path_regrid + output_filename)

    print(f"[INFO] Regridding of {variable} complete. Saved to: {output_path_regrid + output_filename}")


Regridding g200 complete
Regridding psl complete


## 2. Anomalies regridded data ERA5
This script processes the regridded ERA5 datasets to compute standardized anomalies
for selected atmospheric variables. The anomalies are calculated by removing
the seasonal climatology and applying a LOESS smoothing (30-day window).

Workflow:
1. Load the regridded 1º x 1º ERA5 dataset for each variable.
2. Remove February 29th to maintain consistent day counts across years.
3. Compute anomalies using `Compute_anomalies`, optionally referencing a baseline period (1950–2000).
4. Drop the original variable from the dataset to reduce file size.
5. Add the computed anomalies as a new variable with descriptive metadata.
6. Save the dataset containing only anomalies to a NetCDF file.

This prepares the data for downstream modeling while keeping storage efficient.

In [3]:
for var in variables_anomalies:
    file_regridded = f'{var}_regridded_1x1_era5_data_full_domain_1950_2024.nc'

    # Load the regridded dataset
    ds = xr.open_dataset(input_path_anomalies + file_regridded)

    # Remove February 29th to ensure consistent number of days per year
    ds = ds.sel(time=~((ds.time.dt.month == 2) & (ds.time.dt.day == 29)))

    # Compute anomalies relative to reference period 1950–2000
    anomalies = Compute_anomalies(ds, var, '1950', '2000')
    print(f"[INFO] Anomalies for {var} computed successfully")

    # Drop original variable to keep only anomalies
    ds = ds.drop_vars(var)

    # Add the anomaly variable with metadata
    ds[f'{var}_anomalies'] = anomalies
    ds[f'{var}_anomalies'].attrs = {
        'long_name': 'Anomalies with LOESS-fit climatology',
        'description': 'Daily standardized anomalies computed with a 30-day LOESS-fit of the climatology'
    }

    # Save dataset containing only anomalies
    ds.to_netcdf(output_path_anomalies + f"{var}_1x1_era5_data_only_anomalies_1950_2024.nc")
    print(f"[INFO] {var} processed and anomalies saved to NetCDF")

anomalies g200 computed, moving to creating netCDF to be saved
g200 done and anomalies saved to netcdf
anomalies psl computed, moving to creating netCDF to be saved
psl done and anomalies saved to netcdf


## 3. Preparation of time-lagged features for ERA5
This script generates lagged features from the previously computed standardized
anomalies for selected atmospheric variables. Lagged features capture temporal
dependencies, which are essential for predicting extreme events using machine
learning models.

Workflow:
1. Load the NetCDF file containing the standardized anomalies for a given variable.
2. Optionally remove specific dates that may cause inconsistencies (e.g., '2024-07-31').
3. Apply the `create_lagged_features_single` function to generate lagged data according
   to a specified lag dictionary.
4. Save the resulting lagged anomaly dataset to a new NetCDF file.

In [7]:
# Loop over each variable to generate and save lagged features
for var in variables_lagged:
    # Input path: NetCDF file containing standardized anomalies
    filename_input = f'{var}_1x1_era5_data_only_anomalies_1950_2024.nc'

    # Load dataset
    ds = xr.open_dataset(input_path_lagged + filename_input)

    # Remove a specific date if necessary to maintain consistency
    ds = ds.where(ds.time.dt.strftime('%Y-%m-%d') != '2024-07-31', drop=True)

    # Output path: NetCDF file to save lagged anomalies
    filename_output = f'{var}_1x1_lagged_standarized_anomalies.nc'

    # Generate lagged features
    ds_lagged = lagged_data.create_lagged_features_single(ds, var, lag_dict[var], 'lagged_era5')

    # Save the dataset containing lagged anomalies
    ds_lagged.to_netcdf(output_path_lagged + filename_output)

    # Close datasets to free resources
    ds.close()
    ds_lagged.close()

    print(f"[INFO] Lagged anomalies for {var} completed and saved to {output_path_lagged + filename_output}")

g500 completed
g200 completed
psl completed
